## Wave 3 Texts

Intend to analyze the responses of humans and AI in several ways:
- Lexical Analysis
- Semantic Analysis
- Analysis of Lexical and Semantic grouped within human/model, question category, question

In [10]:
# %pip install openai plotly seaborn statsmodels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 80.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 23.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 112.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 22.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
# load necessary packages
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [4]:
dfl = pd.read_csv('./data/wave 3/wave 3 response evals - long v4 - with qr text and w2 actuals.csv')

In [5]:
# construct SDO_ variable that is the average of SDO and reverse-coded SDO_R
dfl['w3_rated_SDO_'] = (dfl['w3_rated_SDO'] + (6 - dfl['w3_rated_SDO_R'])) / 2

In [6]:
modelSort = ['human',
             'fb-opt-1.3b','fb-opt-iml-1.3b',
             'gpt-2','gpt-3','ChatGPT','gpt-4']
# convert model to categorical dtype
from pandas.api.types import CategoricalDtype
dfl['model'] = dfl['model'].astype(CategoricalDtype(categories=modelSort, ordered=False))
dfl.model.dtype

CategoricalDtype(categories=['human', 'fb-opt-1.3b', 'fb-opt-iml-1.3b', 'gpt-2', 'gpt-3',
                  'ChatGPT', 'gpt-4'],
, ordered=False)

In [7]:
dfl.columns

Index(['w3_rater_ResponseId', 'w3_rater_StartDate',
       'w3_rater_LocationLatitude', 'w3_rater_LocationLongitude',
       'w3_rater_gender', 'w3_rater_age', 'w3_rater_height', 'w3_rater_weight',
       'w3_rater_race', 'w3_rater_employ', 'w3_rater_income', 'w3_rater_educ',
       'w3_rater_poli_affil', 'w3_rater_voted', 'w3_rater_big5_extra',
       'w3_rater_big5_agree', 'w3_rater_big5_conc', 'w3_rater_big5_neuro',
       'w3_rater_big5_open', 'w3_rater_SDO_R', 'w3_rater_SDO', 'Rid', 'Index',
       'w3_rated_gender', 'w3_rated_age', 'w3_rated_height', 'w3_rated_weight',
       'w3_rated_race', 'w3_rated_income_19', 'w3_rated_education',
       'w3_rated_conservative', 'w3_rated_voted', 'w3_rated_extravert',
       'w3_rated_agreeable', 'w3_rated_concientious', 'w3_rated_neurotic',
       'w3_rated_open', 'w3_rated_SDO_R', 'w3_rated_SDO', 'w3_rated_trust',
       'w3_rated_interact', 'model', 'index', 'qcat', 'qid', 'question',
       'response', 'w2_ResponseId', 'w3_rated_SDO_', '

In [11]:
# create ada embeddings for all the responses
import json
import openai
from openai.embeddings_utils import (
    get_embedding, cosine_similarity,
    tsne_components_from_embeddings,
    chart_from_components,
)
import os

In [12]:
dfl.shape

(38605, 63)

In [13]:
# load config
with open(r'config.json') as config_file:
    config_details = json.load(config_file)
    
# Setting up the deployment name
deployment_name = config_details['EMBEDDINGS_MODEL']

# This is set to `azure`
openai.api_type = "azure"

# The API key for your Azure OpenAI resource.
openai.api_key = config_details["OPENAI_API_KEY"]

# The base URL for your Azure OpenAI resource. e.g. "https://<your resource name>.openai.azure.com"
openai.api_base = config_details['OPENAI_API_BASE']

# Currently OPENAI API have the following versions available: 2022-12-01
openai.api_version = config_details['OPENAI_API_VERSION']

In [29]:
# test embedding on subset of data
dfs = dfl[['model', 'index', 'qcat', 'qid', 'question','response',]].head()

In [11]:
dfs

,model,index,qcat,qid,question,response
0,fb-opt-1.3b,1,1,1,What is your opinion on abortion?,"I agree that abortion is a serious matter, but..."
1,fb-opt-1.3b,2,1,1,What is your opinion on abortion?,"I have no opinion, but I am very concerned wit..."
2,fb-opt-1.3b,3,1,1,What is your opinion on abortion?,"If you're planning on having kids, have them. ..."
3,fb-opt-1.3b,4,1,1,What is your opinion on abortion?,I don't think you can say that abortion should...
4,fb-opt-1.3b,5,1,1,What is your opinion on abortion?,"I have no opinion, but I am very concerned wit..."


In [30]:
dfs['response_ada_v2'] = dfs["response"].apply(lambda x : get_embedding(x, engine = 'text-embedding-ada-002')) # engine should be set to the deployment name you chose when you deployed the text-embedding-ada-002 (Version 2) model

In [34]:
dfs[['response','response_ada_v2']].head()

,response,response_ada_v2
0,"I agree that abortion is a serious matter, but...","[-0.02235282212495804, -0.03829408064484596, -..."
1,"I have no opinion, but I am very concerned wit...","[-0.027138056233525276, -0.025159239768981934,..."
2,"If you're planning on having kids, have them. ...","[-0.011410890147089958, -0.01900525577366352, ..."
3,I don't think you can say that abortion should...,"[-0.008020703680813313, -0.02725287526845932, ..."
4,"I have no opinion, but I am very concerned wit...","[-0.027138056233525276, -0.025159239768981934,..."


In [56]:
# try collecting embeddings for all responses
# collect embeddings for all human responses
hdfl = dfl[dfl['model'] == 'human']
hdfl.shape


(5513, 63)

In [57]:
hdfl['response_ada_v2'] = hdfl["response"].apply(
    lambda x : get_embedding(
        x, 
        engine = 'text-embedding-ada-002')) # engine should be set to the deployment name you chose when you deployed the text-embedding-ada-002 (Version 2) model

RetryError: RetryError[<Future at 0x1fcc724fbe0 state=finished raised RateLimitError>]

(38605,)

In [66]:
L

38605

In [14]:
response_list = dfl['response'].tolist()
len(response_list)

38605

In [77]:
response_list[577]

'Police brutality is an egregious violation of the rights of individuals and should not be tolerated in any form. The police are responsible for protecting and serving the public, and any act of brutality is a violation of that duty. There needs to be greater accountability and transparency in law enforcement, and those found to have engaged in brutality should be held to the highest standards of justice.'

In [15]:
import datetime
st = datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
print(st)

2023-11-01-10-10-04


In [16]:
import time 
import datetime
st = datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
print(st)

embed_list = []
i = 0
response_list = dfl['response'].tolist()
L = len(response_list)
while i < L:
    r = response_list[i]
    if i % 100 == 0: print(f'{i}: {r}')
    try:
        embed = get_embedding(r, engine = 'text-embedding-ada-002')
        embed_list.append(embed)
        i += 1 # increment counter only if successful
    except Exception as e:
        print(e)
        time.sleep(2)
    time.sleep(0.05)

embeddings = np.array(embed_list)
print(embeddings.shape)
np.save(f'./data/wave 3/response_embeddings-{st}.npy', embeddings)


2023-11-01-10-11-05
0: I agree that abortion is a serious matter, but I am also a pro-choice person, and not a pro-abortion person.
I think that women should be able to make decisions for themselves about their bodies, and that the government should not have a say in it.
Obviously, there are some cases where women should have the right to have an abortion.
100: The question is not whether one should be pro-life or pro-choice. The question is whether one should be pro-life or pro-choice, or pro-pro-life or pro-pro-choice."
This is not a choice. It is a decision.
200: A fetus typically becomes viable at 24 weeks of gestation, although it can vary depending on the circumstances.
300: My understanding of the issue of abortion is that it is a highly contentious and divisive issue that is deeply personal and varies depending on individual beliefs and values. I believe that women should have the right to choose whether or not to terminate a pregnancy and that the decision should be made betwe

In [62]:
i

356

In [41]:
# test saving and reloading
# remove the embedding column
dfs_embeddings = dfs['response_ada_v2']
# dfs = dfs.drop(columns=['response_ada_v2'])

# save the embeddings
np_df_embeddings = np.array(dfs_embeddings.to_list())
np.save('./data/wave 3/response_embeddings.npy', np_df_embeddings)
# save the csv
# dfs.to_csv('./data/wave 3/dfs-test.csv', index=False)

In [46]:
np_df_embeddings = np.load('./data/wave 3/response_embeddings.npy')

In [51]:
np_df_embeddings

array([[-0.02235282, -0.03829408, -0.01292331, ..., -0.01967299,
        -0.01066925, -0.04012238],
       [-0.02713806, -0.02515924,  0.00655965, ..., -0.00756833,
        -0.01453274, -0.04528149],
       [-0.01141089, -0.01900526, -0.00268994, ..., -0.01612997,
         0.02228025, -0.01765142],
       [-0.0080207 , -0.02725288,  0.00807075, ..., -0.01037937,
         0.01024173, -0.03165738],
       [-0.02713806, -0.02515924,  0.00655965, ..., -0.00756833,
        -0.01453274, -0.04528149]])

In [26]:
dfs = pd.read_csv('./data/wave 3/dfs-test.csv')

In [52]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_mean_cosine_sim(dfCol):
    if type(dfCol) == np.ndarray:
        vectors = dfCol
    # convert vectors column to numpy array
    else:
        vectors = np.array(dfCol.tolist())

    # calculate cosine similarity between all pairs of vectors
    cos_sim = cosine_similarity(vectors)

    # calculate average cosine similarity between all pairs of vectors
    return cos_sim[np.tril_indices_from(cos_sim, k=-1)].mean()

In [53]:
get_mean_cosine_sim(dfs['response_ada_v2'])

0.8290292613560318

In [54]:
get_mean_cosine_sim(np_df_embeddings)

0.8290292613560318

In [24]:
vectors = np.array(dfs['response_ada_v2'].tolist())
vectors

array([[-0.02235282, -0.03829408, -0.01292331, ..., -0.01967299,
        -0.01066925, -0.04012238],
       [-0.02713806, -0.02515924,  0.00655965, ..., -0.00756833,
        -0.01453274, -0.04528149],
       [-0.01141089, -0.01900526, -0.00268994, ..., -0.01612997,
         0.02228025, -0.01765142],
       [-0.0080207 , -0.02725288,  0.00807075, ..., -0.01037937,
         0.01024173, -0.03165738],
       [-0.02713806, -0.02515924,  0.00655965, ..., -0.00756833,
        -0.01453274, -0.04528149]])

In [16]:
vectors.shape

(5, 1536)

In [19]:
# average across all vectors
vmean = vectors.mean(axis=0)
vmean.shape

(1536,)

In [20]:
# calculate cosine similarity between v1 and all vectors in vectors
cos_sim = cosine_similarity(vmean.reshape(1, -1), vectors)

# get index of closest vector
closest_idx = cos_sim.argmax()

print(closest_idx)

1


In [ ]:
# get all embeddings (possibly done above)

In [ ]:
# actuals variables
hcol = [
    # id
    # 'w2_ResponseId', 

# gender
'w2_actual_gender', # keep male female only

# big 5
'w2_actual_open','w2_actual_concientious', 
'w2_actual_extravert', 'w2_actual_agreeable',
'w2_actual_neurotic', 

# keep categories
'w2_actual_race', 'w2_actual_education',
    
# convert to quintiles and save labels of midpoints
'w2_actual_SDO', 'w2_actual_conservative', # 5 point scale
'w2_actual_income', 'w2_actual_age',
'w2_actual_weight', 'w2_actual_height'
]

In [ ]:
# create quintiles of numeric variables

In [ ]:
# for each attribute
# for each level

In [21]:
cos_sim

array([[0.93721715, 0.95036267, 0.87802806, 0.92952031, 0.95036267]])

In [ ]:
# calculate cosine similarity between all vectors in vectors with the vmean
cos_sim = cosine_similarity(vectors, vmean.reshape(1, -1))


In [18]:
vmean.reshape(1, -1).shape

(1, 1536)

In [34]:
cos_sim = cosine_similarity(vectors)

In [ ]:
# 

In [1]:
# test aggregation on column of vectors
dfs[['model','response_ada_v2']].groupby('model').response_ada_v2.agg(lambda x: get_mean_cosine_sim(x))

NameError: name 'dfs' is not defined

In [32]:
dfs.groupby('model').apply(get_mean_cosine_sim, col='response_ada_v2')

C:\Users\niswitan\AppData\Local\Temp\ipykernel_17392\1545653399.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dfs.groupby('model').apply(get_mean_cosine_sim, col='response_ada_v2')


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [19]:
cos_sim

array([[1.        , 0.84115586, 0.77392315, 0.8975988 , 0.84115586],
       [0.84115586, 1.        , 0.75798983, 0.81575538, 1.        ],
       [0.77392315, 0.75798983, 1.        , 0.78896851, 0.75798983],
       [0.8975988 , 0.81575538, 0.78896851, 1.        , 0.81575538],
       [0.84115586, 1.        , 0.75798983, 0.81575538, 1.        ]])

In [20]:
# mean of values of lower triangle of cos_sim
cos_sim_df.values[np.tril_indices_from(cos_sim_df.values, k=-1)].mean()

0.8290292613560318

In [27]:
# mean of values of lower triangle of cos_sim
cos_sim[np.tril_indices_from(cos_sim, k=-1)].mean()

0.8290292613560318

In [21]:
import numpy as np

# extract the lower triangle of the array
lower_triangle = np.tril(cos_sim)

# calculate the average of the values in the lower triangle
avg_lower_triangle = np.mean(lower_triangle)

print(avg_lower_triangle)

0.5316117045424128


In [ ]:
# given set of vectors, calculate the cosine similarities between all pairs
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity
dfs['response_ada_v2'] = dfs['response_ada_v2'].apply(lambda x: np.array(x))
dfs['response_ada_v2'] = dfs['response_ada_v2'].apply(lambda x: x.reshape(1,-1))
dfs['response_ada_v2'] = dfs['response_ada_v2'].apply(lambda x: x[0])
dfs['response_ada_v2']


In [16]:
dfs['response_tsne_comp'] = dfs["response_ada_v2"].apply(
    lambda x : tsne_components_from_embeddings(x)
    )

ValueError: Expected 2D array, got 1D array instead:
array=[-0.02235282 -0.03829408 -0.01292331 ... -0.01967299 -0.01066925
 -0.04012238].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.